# 08 · Vector Databases

> **Source notes:** `VectorDatabases.md`

This notebook demonstrates **vector databases** (ChromaDB) and compares them to the in-memory pattern from Chapter 02. You will learn:
- Creating + querying collections
- Metadata filtering + distance metrics
- Hybrid search (keyword + semantic)
- When to use ChromaDB vs Pinecone vs in-memory search
- Production patterns (persistence, multi-tenancy, indexing)

All demos use **ChromaDB** (local, no cloud dependencies). Running example: *product recommendation system*.

In [ ]:
# Import required libraries
import numpy as np
import time
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
import chromadb
from chromadb.config import Settings

print("✓ Libraries loaded")
print(f"  NumPy: {np.__version__}")
print(f"  ChromaDB: {chromadb.__version__}")

# Check for FAISS (install if needed: pip install faiss-cpu)
try:
    import faiss
    print(f"  FAISS: {faiss.__version__}")
except ImportError:
    print("  ⚠ FAISS not installed (optional but recommended)")
    print("    Install: pip install faiss-cpu")


## 1 · Distance Metrics — How "Closeness" Is Measured

All vector search rests on a **distance (or similarity) function**:

| Metric | Formula | Use When |
|--------|---------|----------|
| **Euclidean (L2)** | `||a - b||` | Image embeddings, sensor data |
| **Cosine Similarity** | `(a · b) / (||a|| · ||b||)` | Text / semantic similarity |
| **Dot Product** | `a · b` | Recommendation; magnitude matters |

**Interview-critical fact:** when vectors are **L2-normalised** (unit length), cosine similarity and dot product give identical rankings, and maximising dot product equals minimising Euclidean distance.

Most embedding models (sentence-transformers with `normalize_embeddings=True`) output unit vectors, making the metric choice less critical in practice — but knowing *why* matters.

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

# Same 3 vectors, test different metrics
v1 = np.array([1.0, 2.0, 3.0])
v2 = np.array([2.0, 3.0, 4.0])
v3 = np.array([10.0, 1.0, 1.0])

vectors = np.array([v1, v2, v3])

# L2 (Euclidean)
l2_from_v1 = euclidean_distances([v1], vectors)[0]

# Cosine similarity
cos_from_v1 = cosine_similarity([v1], vectors)[0]

# Dot product
dot_from_v1 = np.dot(vectors, v1)

print("Query vector: v1 = [1.0, 2.0, 3.0]\n")
print("L2 distance (lower = closer):")
for i, dist in enumerate(l2_from_v1):
    print(f"  v{i+1}: {dist:.3f}")

print("\nCosine similarity (higher = closer):")
for i, sim in enumerate(cos_from_v1):
    print(f"  v{i+1}: {sim:.3f}")

print("\nDot product (higher = closer):")
for i, dot in enumerate(dot_from_v1):
    print(f"  v{i+1}: {dot:.3f}")

# Now L2-normalize and repeat
vectors_norm = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
v1_norm = v1 / np.linalg.norm(v1)

cos_norm = cosine_similarity([v1_norm], vectors_norm)[0]
dot_norm = np.dot(vectors_norm, v1_norm)
l2_norm = euclidean_distances([v1_norm], vectors_norm)[0]

print("\n" + "="*60)
print("After L2 normalization:")
print("\nCosine similarity:")
for i, sim in enumerate(cos_norm):
    print(f"  v{i+1}: {sim:.3f}")
print("\nDot product:")
for i, dot in enumerate(dot_norm):
    print(f"  v{i+1}: {dot:.3f}")
print("\n✓ Cosine and dot product now identical")


## 2 · Why Brute-Force Search Fails at Scale

**Exact search** = compute distance to **every** stored vector.

**Time complexity:** `O(N x d)` per query.

| Scale | Dimensions | Ops/query | Time at 10 GFLOP/s |
|-------|-----------|-----------|---------------------|
| 100 K | 384 | 38 M | ~4 ms OK |
| 10 M | 768 | 7.7 B | ~770 ms TOO SLOW |
| 100 M | 768 | 77 B | ~7.7 s WAY TOO SLOW |

**Memory:** 100 M vectors x 768 dims x 4 bytes (float32) = **307 GB** — exceeds most server RAM.

Traditional indexes (kd-trees, ball-trees) degrade in high dimensions — "the curse of dimensionality."

**Solution: Approximate Nearest Neighbour (ANN) indexes** — trade a tiny bit of recall for massive speed gains.

In [ ]:
import numpy as np
import time
from sklearn.metrics.pairwise import cosine_similarity

dims = 384
sizes = [1000, 10000, 100000]

for size in sizes:
    # Generate random corpus
    corpus = np.random.randn(size, dims).astype('float32')
    query = np.random.randn(1, dims).astype('float32')

    # Time brute-force search
    start = time.time()
    scores = cosine_similarity(query, corpus)[0]
    top_10 = np.argsort(scores)[-10:][::-1]
    elapsed = time.time() - start

    print(f"{size:>6,} vectors: {elapsed*1000:>6.1f} ms")

print("\n✓ Observe O(N) growth — brute-force fails at scale")


## 3 · IVF — Inverted File Index

**Idea:** partition the vector space into **K clusters** (k-means). At query time only search the closest `nprobe` clusters instead of everything.

```
Training:
  K-means on corpus --> K centroids + N inverted lists

Query:
  1. Compute distance from query to all K centroids  (cheap: O(K * d))
  2. Pick nprobe closest centroids
  3. Search only the vectors in those clusters
```

**Key tunables:**
- `K` — number of clusters (more clusters = finer partitioning = faster, but lower recall per cluster)
- `nprobe` — clusters searched at query time (higher = better recall, slower)

In [ ]:
import numpy as np
import chromadb
from chromadb.config import Settings

# Create in-memory ChromaDB
client = chromadb.Client(Settings(anonymized_telemetry=False))
collection = client.create_collection("test_ivf")

# Generate corpus
n_docs = 10000
dims = 384
corpus = np.random.randn(n_docs, dims).astype('float32')

# Add to collection
collection.add(
    ids=[f"doc_{i}" for i in range(n_docs)],
    embeddings=corpus.tolist()
)

# Test query
query = np.random.randn(1, dims).astype('float32')

# Note: ChromaDB uses HNSW by default, not IVF
# This demonstrates the principle with HNSW ef parameter instead
# (IVF would require FAISS directly)

print("ChromaDB uses HNSW, not IVF")
print("To test IVF nprobe sweep, install faiss-cpu:")
print("  pip install faiss-cpu\n")

print("HNSW equivalent demonstration:")
results_10 = collection.query(
    query_embeddings=query.tolist(),
    n_results=10
)
print(f"Retrieved {len(results_10['ids'][0])} results (n_results=10)")

# For actual IVF demo (if faiss installed):
try:
    import faiss
    index = faiss.IndexIVFFlat(faiss.IndexFlatL2(dims), dims, 100)  # 100 clusters
    index.train(corpus)
    index.add(corpus)

    for nprobe in [1, 4, 16]:
        index.nprobe = nprobe
        start = time.time()
        D, I = index.search(query, 10)
        elapsed = time.time() - start
        print(f"nprobe={nprobe:>2}: {elapsed*1000:>5.1f} ms")
except ImportError:
    print("\nFAISS not installed — skipping IVF demo")


## 4 · HNSW — Hierarchical Navigable Small World

HNSW is a **graph-based** ANN index used by most production vector DBs (ChromaDB, Pinecone, Qdrant, Weaviate, Azure AI Search).

```
Layer 2 (sparse long-range links)   A ----------- B
                                    |             |
Layer 1                             A --- C --- D-B
                                    |             |
Layer 0 (dense local links)         A-c-C-d-D-e-E-B  <- most vectors live here
```

**How search works:**
1. Enter at a random node in the **top (sparse) layer**
2. Greedily navigate toward the query (always step to the closer neighbour)
3. Drop to the layer below when stuck; repeat
4. At Layer 0, do a local exhaustive search among found neighbours

**Key parameters:**
- `M` — max edges per node (higher = better recall, more memory, slower build)
- `ef_construction` — size of dynamic list during build (higher = better quality graph)
- `ef` — size of candidate list at query time (higher = better recall, slower query)

ChromaDB uses HNSW internally with sensible defaults. We just configure the space (metric).

In [ ]:
import numpy as np
import chromadb
from chromadb.config import Settings
import time

# Create collection with HNSW
client = chromadb.Client(Settings(anonymized_telemetry=False))
collection = client.create_collection(
    "test_hnsw",
    metadata={"hnsw:space": "cosine"}
)

# Generate corpus
n_docs = 50000
dims = 384
corpus = np.random.randn(n_docs, dims).astype('float32')

print("Adding 50,000 vectors to ChromaDB (HNSW)...")
collection.add(
    ids=[f"doc_{i}" for i in range(n_docs)],
    embeddings=corpus.tolist()
)

# Test query
query = np.random.randn(1, dims).astype('float32')

# ChromaDB doesn't expose efSearch parameter directly
# But we can observe query time differences
print("\nQuerying with different n_results (proxy for efSearch):")

for n in [10, 50, 100]:
    start = time.time()
    results = collection.query(
        query_embeddings=query.tolist(),
        n_results=n
    )
    elapsed = time.time() - start
    print(f"n_results={n:>3}: {elapsed*1000:>6.1f} ms")

print("\n✓ Higher n_results → more candidates checked → slower but higher recall")


## 5 · DiskANN, Quantization & Index Selection

### DiskANN
- Stores the HNSW graph primarily on **SSD**, caching hot nodes in RAM
- Enables billion-scale search on commodity servers
- Used in Azure AI Search

### Product Quantization (PQ)
Compress vectors by splitting each high-dim vector into M sub-vectors, each quantised to a codebook:

| Original | PQ compressed | Compression |
|---------|--------------|-------------|
| 768 × 4 B = 3072 B | ~96 B | ~32x |

Trade-off: memory savings at the cost of some recall loss.

### Index Selection Guide

| Scenario | Recommended |
|----------|------------|
| < 100 K vectors, perfect recall | **Flat** (brute-force) |
| 100 K – 10 M, general purpose | **HNSW** |
| Large corpus, memory-constrained | **IVF + PQ** |
| Billions of vectors, commodity hardware | **DiskANN** |
| Streaming inserts (no rebuild) | **HNSW** (supports incremental adds) |

## 6 · Key Takeaways

| Concept | One-Liner |
|---------|-----------|
| L2 / Cosine / Dot | On normalised vectors they give identical rankings |
| Brute-force | Perfect recall; O(N*d) cost; impractical above ~100 K |
| IVF | K-means partitioning; tune nprobe for recall vs. speed |
| HNSW | Graph-based greedy descent; dominant in production |
| DiskANN | HNSW for billion-scale; data lives on SSD |
| PQ | Compress vectors 32x; small recall loss |

**Next:** `04-react-agents/notebook.ipynb` — building an agent that uses all of this.

## 5 · FAISS — Real-World ANN (EXERCISE)

**TODO:** Implement FAISS index comparison demonstrating:
1. **IVF with nprobe tuning** — show how recall improves as nprobe increases
2. **HNSW with efSearch tuning** — demonstrate recall/speed tradeoff
3. **SQ8 compression** — 4× memory savings with <1% recall loss
4. **IVF-PQ compression** — 32× memory savings for extreme cases
5. **Memory comparison table** — show bytes per vector for each method

**Hints:**
- Use `faiss.IndexIVFFlat()` for IVF clustering
- Use `faiss.IndexHNSWFlat()` for graph-based HNSW
- Use `faiss.IndexScalarQuantizer()` for SQ8
- Use `faiss.IndexIVFPQ()` for Product Quantization
- Measure `recall@10` by comparing results to ground truth (brute-force)
- Track memory: `corpus.nbytes / 1024**2` for MB

**Expected output:**
```
Index Type                Memory (MB)    Build (s)  Query (ms)  Recall@10
================================================================================
Flat (brute-force)              76.3         0.0        2.34       100%
IVF (nprobe=1)                  76.3         0.5        0.12        45%
IVF (nprobe=16)                 76.3         0.5        0.98        92%
HNSW (ef=16)                   114.5         3.2        0.15        85%
HNSW (ef=64)                   114.5         3.2        0.42        99%
SQ8 (4× compression)            19.1         0.3        1.85        98%
IVF-PQ (32× compression)         2.4         1.1        1.23        87%
```


In [ ]:
# TODO: Import FAISS and implement index comparison
# import faiss

# 1. Generate corpus (50K vectors, 384 dims)
# TODO: Create normalized corpus

# 2. Ground truth (brute-force)
# TODO: Use IndexFlatIP for exact search

# 3. Compare indexes:
#    - IVF with different nprobe values
#    - HNSW with different efSearch values
#    - SQ8 (scalar quantization)
#    - IVF-PQ (product quantization)

# 4. Print comparison table
print("Implement FAISS index comparison here")


## 6 · PizzaBot Connection — Index Choice for a Small Corpus

> Full system spec: [AIPrimer.md](../AIPrimer.md)

The PizzaBot menu corpus (~500 chunks) is small enough that the index choice is mostly invisible to users — but it makes the tradeoffs concrete:

| Scenario | Index | Why |
|---|---|---|
| **Prototype** | FAISS Flat (brute-force, exact) | 500 vectors; exact search < 1 ms; no tuning needed |
| **10k menu items** | FAISS `IndexHNSWFlat` | Sub-millisecond search; `efSearch=64` balances recall/latency |
| **Multi-store (1M+ items)** | IVF-PQ in a dedicated vector DB | Corpus distribution across stores matches IVF cluster assumptions |
| **Allergen safety queries** | Flat or `efSearch=128+` | Safety-critical: favour recall over latency — never miss a allergen flag |

**Key insight applied here:** cosine similarity is the right metric for MiniLM embeddings because they are not L2-normalised by default. The FAISS L2 metric would give different (incorrect) results.

**Try it:** add a FAISS cell that embeds the PizzaBot `allergens.csv` rows and queries `"dairy-free options"` — observe that it returns items without dairy even when the query phrasing doesn't match any row verbatim.
